# Exploring GitSkills

I created this notebook to document how I'm exploring GitSkills, understand the dataset, and see how we might use it in our research.

[GitSkills sample](https://github.com/giuseppedestefanis/gitskills-sample) · July 2026

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

path = Path("../data/gitskills/agent_skills_sample.db")

conn = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)

def query(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

## Tables

In [2]:
query("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'")

,name
0,mining_runs
1,repos
2,artifact_siblings
3,artifacts


Let's start by inspecting the tables. `artifacts` contains the skill files, `repos` contains repository metadata, and `artifact_siblings` contains files and directories accompanying a representative skill. One skill can have many sibling entries.

In [19]:
query("PRAGMA table_info(artifacts)")[["name", "type"]]

,name,type
0,repo_full_name,TEXT
1,path,TEXT
2,filename,TEXT
3,location_class,TEXT
4,file_sha,TEXT
...,...,...
25,first_commit_author_type,TEXT
26,first_commit_message,TEXT
27,last_commit_author,TEXT
28,last_commit_author_type,TEXT


In [21]:
query("PRAGMA table_info(artifact_siblings)")[["name", "type"]]

,name,type
0,repo_full_name,TEXT
1,artifact_path,TEXT
2,entry_name,TEXT
3,entry_type,TEXT
4,entry_size,INTEGER
5,entry_sha,TEXT
6,content,TEXT
7,content_fetched,INTEGER
8,skipped_reason,TEXT


We can link `artifacts` and `artifact_siblings` using `repo_full_name` together with the skill path (`path` in `artifacts` and `artifact_path` in `artifact_siblings`).

## Counts

Let's see how many skill files are duplicates.

In [22]:
query("""
SELECT COUNT(*) AS files, COUNT(DISTINCT file_sha) AS unique_contents
FROM artifacts
""")

,files,unique_contents
0,29786,13000


There are many duplicate files. We could use distinct contents when analysing the skills themselves. To study how they change over time, we would also need their version histories.

## Skills

Appendix A of the paper (p. 3) explains the deduplication process:

> We group files by content hash and select one representative per group for enrichment, preferring a file in the .claude/skills/ directory, with a deterministic rule to break ties. We do not assume that the representative is the original source. All copies remain in the dataset with their repository, path, and location class, so the spread of each content can be measured.

To analyse skill content, let's use one representative per distinct content (`dedup_primary = 1`).

In [25]:
skills = query("""
SELECT name, repo_full_name, path, file_sha, content
FROM artifacts
WHERE dedup_primary = 1 AND content IS NOT NULL
ORDER BY file_sha
LIMIT 10
""")
skills[["name", "repo_full_name", "path"]]

,name,repo_full_name,path
0,None,alas-poor-ophelia/windrose-md,.claude/skills/obsidian-bridge/SKILL.md
1,testing-delivery-gate,etendosoftware/etendo_schema_forge,.agents/skills/testing-delivery-gate/SKILL.md
2,draft-release-blog,obot-platform/obot,.claude/skills/draft-release-blog/SKILL.md
3,fachanwalt-gewerblicher-rechtsschutz-uwg-einstweilige-verfuegung,Klotzkette/claude-fuer-deutsches-recht,fachanwalt-gewerblicher-rechtsschutz/skills/fachanwalt-gewerblicher-rechtsschutz-uwg-einstweilig...
4,writing-skills,OTOJulian/oto-hybrid-framework,oto/skills/writing-skills/SKILL.md
5,token-lifecycle,okfriansyah-moh/crypto-sniping-bot,.github/skills/token-lifecycle/SKILL.md
6,trinity,frankyxhl/trinity,SKILL.md
7,navan-migration-deep-dive,jukeyman/jukeyman-skills,skills/saas-packs--navan-pack--navan-migration-deep-dive/SKILL.md
8,executing-tasks,Nodman/conveyor,plugin/skills/executing-tasks/SKILL.md
9,stop_spotify,rolandknight/voice-chatbot,skills/spotify/stop_spotify/SKILL.md


## Read a skill

In [37]:
skill = skills.iloc[4]
print(skill["content"])

---
name: writing-skills
description: Use when creating new skills, editing existing skills, or verifying skills work before deployment
---

# Writing Skills

## Overview

**Writing skills IS Test-Driven Development applied to process documentation.**

**Personal skills live in agent-specific directories (`~/.claude/skills` for Claude Code, `~/.agents/skills/` for Codex)** 

You write test cases (pressure scenarios with subagents), watch them fail (baseline behavior), write the skill (documentation), watch tests pass (agents comply), and refactor (close loopholes).

**Core principle:** If you didn't watch an agent fail without the skill, you don't know if the skill teaches the right thing.

**REQUIRED BACKGROUND:** You MUST understand oto:test-driven-development before using this skill. That skill defines the fundamental RED-GREEN-REFACTOR cycle. This skill adapts TDD to documentation.

**Official guidance:** For Anthropic's official skill authoring best practices, see anthropic-best-p

## Identical copies

In [47]:
query("""
SELECT repo_full_name, path
FROM artifacts
WHERE file_sha = ?
LIMIT 20
""", (skill["file_sha"],))

,repo_full_name,path
0,OTOJulian/oto-hybrid-framework,oto/skills/writing-skills/SKILL.md


This skill has only one occurrence in the sample. Other skills may have identical copies in different repositories.

## Accompanying files

In [48]:
query("""
SELECT entry_name, entry_type, entry_size
FROM artifact_siblings
WHERE repo_full_name = ? AND artifact_path = ?
LIMIT 20
""", (skill["repo_full_name"], skill["path"]))

,entry_name,entry_type,entry_size
0,anthropic-best-practices.md,file,45820
1,examples,dir,0
2,examples/CLAUDE_MD_TESTING.md,file,5423
3,graphviz-conventions.dot,file,5970
4,persuasion-principles.md,file,5908
5,render-graphs.js,file,4857
6,testing-skills-with-subagents.md,file,12550


A skill can have accompanying files and directories.

## History

In [49]:
query("""
SELECT name, path, first_commit_at, last_commit_at, commit_count
FROM artifacts
WHERE dedup_primary = 1 AND history_fetched = 1 AND commit_count > 1
ORDER BY commit_count DESC
LIMIT 10
""")

,name,path,first_commit_at,last_commit_at,commit_count
0,None,.claude/skills/ac27-level-editor/SKILL.md,2026-06-01T23:02:10Z,2026-07-23T06:35:00Z,70
1,loop-video,.claude/skills/loop-video/SKILL.md,2026-04-09T08:06:48Z,2026-07-22T14:08:09Z,40
2,plan-review,skills/plan-review/SKILL.md,2026-03-25T04:06:32Z,2026-06-30T02:53:40Z,39
3,refine,.claude/skills/refine/SKILL.md,2026-01-19T14:27:57Z,2026-06-04T08:22:51Z,32
4,gabb-code-navigation,.claude/skills/gabb/SKILL.md,2025-12-17T13:20:01Z,2026-04-02T08:06:39Z,31
5,next-task,.claude/skills/next-task/SKILL.md,2026-06-12T13:10:45Z,2026-07-11T00:09:12Z,28
6,auto-respawn,autonomys/auto-respawn/SKILL.md,2026-02-25T12:03:55Z,2026-07-01T09:34:38Z,26
7,run,skills/run/SKILL.md,2026-02-20T07:26:36Z,2026-07-17T14:55:21Z,26
8,None,claude-config/.claude/skills/operationalize-design/skill.md,2026-05-09T22:53:28Z,2026-07-14T08:09:14Z,25
9,codex-speak,skills/codex-speak/SKILL.md,2026-05-30T04:28:16Z,2026-06-13T02:40:24Z,23


## Collection runs

In [46]:
query("SELECT * FROM mining_runs")

,run_id,artifact_type,query,started_at,finished_at,discovered,note
0,1,agent-skills,filename:SKILL.md,2026-07-09T17:26:45.768703+00:00,2026-07-09T18:04:34.751737+00:00,11796,None
1,2,agent-skills,filename:SKILL.md,2026-07-09T18:07:39.150068+00:00,2026-07-12T06:54:51.308396+00:00,1328473,None
2,3,agent-skills,filename:SKILL.md,2026-07-12T06:59:13.601456+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before finishing (repair-runs)
3,4,agent-skills,filename:SKILL.md,2026-07-13T12:52:52.309967+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before finishing (repair-runs)
4,5,agent-skills,filename:SKILL.md,2026-07-14T18:04:23.050325+00:00,2026-07-20T05:23:31.638481+00:00,1657948,None
5,6,agent-skills,filename:SKILL.md,2026-07-20T05:28:44.667482+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before finishing (repair-runs)
6,7,agent-skills,filename:SKILL.md,2026-07-20T07:38:28.581854+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before finishing (repair-runs)


In [12]:
conn.close()